In [2]:
import pandas as pd
import numpy as np
import json
import re
import unicodedata
from pathlib import Path
from rapidfuzz import fuzz, process

RAW_DATA = Path("../data/raw")
PROCESSED_DATA = Path("../data/processed")
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

In [3]:
race_results = pd.read_csv(RAW_DATA / "race_results_2017_2023.csv")
course_data = pd.read_csv(RAW_DATA / "structured_course_data.csv", index_col=0)

print(f"race_results:  {race_results.shape[0]:,} rows x {race_results.shape[1]} cols")
print(f"course_data:   {course_data.shape[0]:,} rows x {course_data.shape[1]} cols")
print(f"race_results columns: {list(race_results.columns)}")
print(f"course_data columns: {list(course_data.columns)}")

race_results:  769,311 rows x 10 cols
course_data:   8,094 rows x 22 cols
race_results columns: ['Race Name', 'Date', 'Rank', 'Team', 'Name', 'Time', 'UCI points', 'PCS points', 'Team Time Trial', 'TimeAfterTeamTTT']
course_data columns: ['Race Name', 'Distance', 'Street', 'Road', 'Paved', 'Asphalt', 'Path', 'Cycleway', 'Unpaved', 'State Road', 'Cobblestones', 'Unknown', 'Compacted Gravel', 'Off-grid (unknown)', 'Singletrack', 'Access Road', 'Alpine', 'Net Gain', 'Lowest Elevation', 'Highest Elevation', 'Vertical Gain', 'Downhill']


In [4]:
race_results["Date"] = pd.to_datetime(race_results["Date"])

for df in [race_results, course_data]:
    df["Year"] = df["Race Name"].str.extract(r"^(\d{4})").astype(int)
    df["Stage"] = df["Race Name"].str.extract(r"Stage\s+(\d+)")
    df["Race"] = (
        df["Race Name"]
        .str.replace(r"^\d{4}\s+", "", regex=True)
        .str.replace(r"\s*Stage\s+\d+", "", regex=True)
        .str.strip()
    )

print(f"Years covered: {sorted(race_results['Year'].unique())}")
print(f"Date range: {race_results['Date'].min()} -> {race_results['Date'].max()}")

Years covered: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Date range: 2017-01-06 00:00:00 -> 2023-10-17 00:00:00


In [5]:
EXCLUSION_PATTERN = (
    r"Femme|Women|Woman|Ladies|Feminas|Feminin|Femminile|"
    r"Femenina|Femines|Femenino|Famenne|vroumen|Junior|Espoir"
)

before_rr = len(race_results)
before_cd = len(course_data)

race_results = race_results[
    ~race_results["Race Name"].str.contains(EXCLUSION_PATTERN, na=False, case=False)
]
course_data = course_data[
    ~course_data["Race Name"].str.contains(EXCLUSION_PATTERN, na=False, case=False)
]

print(f"race_results: {before_rr:,} -> {len(race_results):,} rows "
      f"(dropped {before_rr - len(race_results):,})")
print(f"course_data:  {before_cd:,} -> {len(course_data):,} rows "
      f"(dropped {before_cd - len(course_data):,})")

race_results: 769,311 -> 703,830 rows (dropped 65,481)
course_data:  8,094 -> 7,319 rows (dropped 775)


In [6]:
# ============================================================
# UCI WORLD TOUR FILTER — all logic in one cell to prevent
# execution order issues
# ============================================================

UCI_WORLD_TOUR_RACES = {
    2017: {
        "Tour Down Under", "Great Ocean Road Race", "Abu Dhabi Tour",
        "Omloop Het Nieuwsblad", "Strade Bianche", "Paris-Nice",
        "Tirreno-Adriatico", "Milan-San Remo", "Volta a Catalunya",
        "Dwars door Vlaanderen", "E3 Harelbeke", "Gent-Wevelgem",
        "Ronde Van Vlaanderen", "Tour of the Basque Country", "Paris-Roubaix",
        "Amstel Gold Race", "La Fleche Wallonne", "Liege-Bastogne-Liege",
        "Tour de Romandie", "Eschborn-Frankfurt",
        "Giro d'Italia", "Tour of California", "Criterium du Dauphine",
        "Tour de Suisse", "Tour de France", "Clasica de San Sebastian",
        "Tour de Pologne", "RideLondon-Surrey Classic", "BinckBank Tour",
        "Vuelta a Espana", "EuroEyes Cyclassics", "Bretagne Classic Ouest-France",
        "GP de Quebec", "GP de Montreal", "Il Lombardia",
        "Presidential Tour of Turkey", "Tour of Guangxi", "Tour of Oman",
    },
    2018: {
        "Tour Down Under", "Great Ocean Road Race", "Abu Dhabi Tour",
        "Omloop Het Nieuwsblad", "Strade Bianche", "Paris-Nice",
        "Tirreno-Adriatico", "Milan-San Remo", "Volta a Catalunya",
        "E3 Harelbeke", "Gent-Wevelgem", "Dwars door Vlaanderen",
        "Ronde Van Vlaanderen", "Tour of the Basque Country", "Paris-Roubaix",
        "Amstel Gold Race", "La Fleche Wallonne", "Liege-Bastogne-Liege",
        "Tour de Romandie", "Eschborn-Frankfurt", "Giro d'Italia",
        "Tour of California", "Criterium du Dauphine", "Tour de Suisse",
        "Tour de France", "Clasica de San Sebastian", "Tour de Pologne",
        "RideLondon-Surrey Classic", "BinckBank Tour", "Vuelta a Espana",
        "EuroEyes Cyclassics", "Bretagne Classic Ouest-France",
        "GP de Quebec", "GP de Montreal", "Presidential Tour of Turkey",
        "Il Lombardia", "Tour of Guangxi", "Tour of Oman",
    },
    2019: {
        "Tour Down Under", "Great Ocean Road Race", "UAE Tour",
        "Omloop Het Nieuwsblad", "Strade Bianche", "Paris-Nice",
        "Tirreno-Adriatico", "Milan-San Remo", "Volta a Catalunya",
        "Classic Brugge-De Panne", "E3 BinckBank Classic",
        "Gent-Wevelgem", "Dwars door Vlaanderen", "Ronde Van Vlaanderen",
        "Tour of the Basque Country", "Paris-Roubaix",
        "Presidential Tour of Turkey", "Amstel Gold Race",
        "La Fleche Wallonne", "Liege-Bastogne-Liege", "Tour de Romandie",
        "Eschborn-Frankfurt", "Giro d'Italia", "Tour of California",
        "Criterium du Dauphine", "Tour de Suisse", "Tour de France",
        "Clasica de San Sebastian", "Tour de Pologne",
        "RideLondon-Surrey Classic", "BinckBank Tour", "Vuelta a Espana",
        "EuroEyes Cyclassics", "Bretagne Classic Ouest-France",
        "GP de Quebec", "GP de Montreal", "Il Lombardia", "Tour of Guangxi",
        "Tour of Oman",
    },
    2020: {
        "Tour Down Under", "Great Ocean Road Race", "UAE Tour",
        "Omloop Het Nieuwsblad", "Paris-Nice", "Strade Bianche",
        "Tour de Pologne", "Milan-San Remo", "Criterium du Dauphine",
        "Il Lombardia", "Bretagne Classic Ouest-France", "Tour de France",
        "Tirreno-Adriatico", "BinckBank Tour", "La Fleche Wallonne",
        "Giro d'Italia", "Liege-Bastogne-Liege", "Gent-Wevelgem",
        "Ronde van Vlaanderen", "Vuelta a Espana",
        "Classic Brugge-De Panne", "Volta a Catalunya"
    },
    2021: {
        "UAE Tour", "Omloop Het Nieuwsblad", "Strade Bianche", "Paris-Nice",
        "Tirreno-Adriatico", "Milan-San Remo", "Classic Brugge-De Panne",
        "E3 Saxo Bank Classic", "Gent-Wevelgem", "Dwars door Vlaanderen",
        "Ronde van Vlaanderen", "Itzulia Basque Country",
        "Paris-Roubaix", "Amstel Gold Race", "La Fleche Wallonne",
        "Liege-Bastogne-Liege", "Tour de Romandie", "Eschborn-Frankfurt",
        "Giro d'Italia", "Criterium du Dauphine", "Tour de Suisse",
        "Tour de France", "Clasica de San Sebastian", "Benelux Tour",
        "Vuelta a Espana", "Bretagne Classic Ouest-France",
        "GP de Quebec", "GP de Montreal", "Il Lombardia", "Tour of Guangxi",
        "Tour de Pologne", "Volta a Catalunya"
    },
    2022: {
        "UAE Tour", "Omloop Het Nieuwsblad", "Strade Bianche", "Paris-Nice",
        "Tirreno-Adriatico", "Milan-San Remo", "Classic Brugge-De Panne",
        "E3 Saxo Bank Classic", "Gent-Wevelgem", "Dwars door Vlaanderen",
        "Ronde van Vlaanderen", "Itzulia Basque Country",
        "Paris-Roubaix", "Amstel Gold Race", "La Fleche Wallonne",
        "Liege-Bastogne-Liege", "Tour de Romandie", "Eschborn-Frankfurt",
        "Giro d'Italia", "Criterium du Dauphine", "Tour de Suisse",
        "Tour de France", "Clasica de San Sebastian", "Vuelta a Espana",
        "Bretagne Classic Ouest-France", "GP de Quebec", "GP de Montreal",
        "Il Lombardia", "Tour de Pologne", "Volta a Catalunya"
    },
    2023: {
        "UAE Tour", "Omloop Het Nieuwsblad", "Strade Bianche", "Paris-Nice",
        "Tirreno-Adriatico", "Milan-San Remo", "Classic Brugge-De Panne",
        "E3 Saxo Bank Classic", "Gent-Wevelgem", "Dwars door Vlaanderen",
        "Ronde van Vlaanderen", "Itzulia Basque Country",
        "Paris-Roubaix", "Amstel Gold Race", "La Fleche Wallonne",
        "Liege-Bastogne-Liege", "Tour de Romandie", "Eschborn-Frankfurt",
        "Giro d'Italia", "Criterium du Dauphine", "Tour de Suisse",
        "Tour de France", "Clasica de San Sebastian", "Vuelta a Espana",
        "Bretagne Classic Ouest-France", "GP de Quebec", "GP de Montreal",
        "Il Lombardia", "Volta a Catalunya"
    },
}

# -------------------------------------------------------------------
# Name aliases — ALL keys/values use plain ASCII so normalize_name
# produces predictable output. No dashes or special chars in keys.
# -------------------------------------------------------------------
NAME_ALIASES = {
    "abu dhabi tour":                                   "uae tour",
    "three days of bruges de panne":                   "classic brugge de panne",
    "bruges de panne":                                 "classic brugge de panne",
    "e3 harelbeke":                                    "e3 saxo bank classic",
    "e3 binckbank classic":                            "e3 saxo bank classic",
    "eschborn frankfurt rund um den finanzplatz":      "eschborn frankfurt",
    "euroeyes cyclassics":                             "hamburg cyclassics",
    # NOTE: BinckBank is NOT aliased to Benelux — it exists under its
    # own name in the 2017-2020 lookup sets directly.
    "ronde van vlaanderen tour des flandres":          "ronde van vlaanderen",
    "la vuelta ciclista a espana":                     "vuelta a espana",
    "la vuelta a espana":                              "vuelta a espana",
    "vuelta espana":                                   "vuelta a espana",
    "presidential cycling tour of turkey":             "presidential tour of turkey",
    "volta ciclista a catalunya":                      "volta a catalunya",
    "giro ditalia":                                    "giro d italia",
    "itzulia basque country":                          "tour of the basque country",
}

FUZZY_BLOCKLIST = {
    "tour de normandie",
    "tour de wallonie",
    "tour du limousin",
    "tour du poitou charentes",
    "tour de bretagne",
    "tour de vendee",
    "tour alsace",
}

FUZZY_THRESHOLD = 88

def normalize_name(name: str) -> str:
    """Lowercase, normalize apostrophes, strip accents, remove punctuation, collapse spaces."""
    name = name.lower()
    name = re.sub(r"[‘’`´']", " ", name)
    name = "".join(
        c for c in unicodedata.normalize("NFD", name)
        if unicodedata.category(c) != "Mn"
    )
    name = re.sub(r"[^a-z0-9]+", " ", name)
    return re.sub(r"\s+", " ", name).strip()

# Build normalized lookups
normalized_uci = {
    year: {normalize_name(r) for r in races}
    for year, races in UCI_WORLD_TOUR_RACES.items()
}

normalized_aliases = {
    normalize_name(k): normalize_name(v)
    for k, v in NAME_ALIASES.items()
}

def is_uci_race(year: int, race_name: str, lookup: dict) -> tuple[bool, str]:
    if pd.isna(year) or pd.isna(race_name):
        return False, "null"
    year = int(year)
    candidates = lookup.get(year, set())
    if not candidates:
        return False, "year_not_in_lookup"
    norm = normalize_name(race_name)
    norm = normalized_aliases.get(norm, norm)
    if norm in candidates:
        return True, "exact"
    if any(blocked in norm for blocked in FUZZY_BLOCKLIST):
        return False, "blocklisted"
    match, score, _ = process.extractOne(norm, candidates, scorer=fuzz.token_sort_ratio)
    if score >= FUZZY_THRESHOLD:
        return True, f"fuzzy:{score:.0f}:{match}"
    return False, f"no_match:{score:.0f}"

# Apply filter
for df_name, df in [("race_results", race_results), ("course_data", course_data)]:
    results = df.apply(
        lambda row: is_uci_race(row["Year"], row["Race"], normalized_uci), axis=1
    )
    df["is_uci"]       = results.apply(lambda x: x[0])
    df["match_reason"] = results.apply(lambda x: x[1])

# Quick verification
print("=== KEY RACE LOOKUPS ===")
checks = [
    (2017, "BinckBank Tour"),
    (2020, "BinckBank Tour"),
    (2021, "Tour de Pologne"),
    (2019, "Tour of Oman"),
    (2023, "Giro d'Italia"),
    (2019, "La Vuelta ciclista a Espana"),
]
for yr, name in checks:
    result, reason = is_uci_race(yr, name, normalized_uci)
    print(f"  {yr} {name:<40} -> {result} ({reason})")

print("=== FUZZY MATCHES (should be empty or legitimate) ===")
fuzzy = race_results[race_results["match_reason"].str.startswith("fuzzy", na=False)]
print(fuzzy[["Race Name", "Year", "match_reason"]].drop_duplicates("Race Name").to_string())

=== KEY RACE LOOKUPS ===
  2017 BinckBank Tour                           -> True (exact)
  2020 BinckBank Tour                           -> True (exact)
  2021 Tour de Pologne                          -> True (exact)
  2019 Tour of Oman                             -> True (exact)
  2023 Giro d'Italia                            -> True (exact)
  2019 La Vuelta ciclista a Espana              -> True (exact)
=== FUZZY MATCHES (should be empty or legitimate) ===
Empty DataFrame
Columns: [Race Name, Year, match_reason]
Index: []


In [7]:
# Review what is being dropped before committing
dropped_rr = race_results[~race_results["is_uci"]]
dropped_cd = course_data[~course_data["is_uci"]]

print(f"Dropping from race_results: {len(dropped_rr):,} rows")
print("Sample dropped races:")
print(dropped_rr["Race"].value_counts().head(20).to_string())

print(f"Dropping from course_data: {len(dropped_cd):,} rows")
print("Sample dropped races:")
print(dropped_cd["Race"].value_counts().head(20).to_string())

Dropping from race_results: 563,257 rows
Sample dropped races:
Race
Tour of Qinghai Lake                            7357
Tour de l'Avenir                                6433
Volta ao Algarve em Bicicleta                   5093
Volta a la Comunitat Valenciana                 4868
Baloise Belgium Tour                            4272
Vuelta al Tachira en Bicicleta                  4230
Tour Cycliste International de la Guadeloupe    4077
Tour of Slovenia                                3927
Vuelta a Burgos                                 3833
PostNord Danmark Rundt - Tour of Denmark        3635
Vuelta a Colombia                               3624
Tour de Hongrie                                 3450
Settimana Internazionale Coppi e Bartali        3342
Vuelta a Andalucia Ruta Ciclista Del Sol        3243
Tour de Normandie                               3161
Giro Ciclistico d'Italia                        3128
Tour of Norway                                  3018
Tour du Rwanda                 

In [9]:
race_results = race_results[race_results["is_uci"]].copy()
course_data  = course_data[course_data["is_uci"]].copy()

# Drop known bad records from prior EDA
# Gent-Wevelgem: wrong distances across multiple years
course_data = course_data[~course_data["Race Name"].str.contains("Wevelgem", na=False)]

# 2023 Tour de Suisse summary row with no stage number
bad_tds = course_data[
    (course_data["Race Name"] == "2023 Tour de Suisse") &
    (course_data["Stage"].isna())
].index
course_data = course_data.drop(bad_tds)

# Liege 2022 duplicate — keep longer distance (257km is correct)
liege_2022 = course_data[
    course_data["Race Name"].str.contains("Liege|Liege", na=False, case=False) &
    (course_data["Year"] == 2022)
]
if len(liege_2022) > 1:
    drop_idx = liege_2022.sort_values("Distance").index[:-1]
    course_data = course_data.drop(drop_idx)

print(f"race_results after filter: {len(race_results):,} rows")
print(f"course_data  after filter: {len(course_data):,} rows")

race_results after filter: 140,573 rows
course_data  after filter: 963 rows


In [10]:
NON_FINISHES = {"DNF", "DNS", "DSQ", "OTL", "DF", "NR"}
BAD_VALUES   = {"1006.0", "10077"}

def clean_rank(val) -> tuple[int, bool]:
    """
    Returns (rank, did_finish).
    DNF/DNS/etc -> (999, False) so they never look like winners
    in any averaging or top-N calculation.
    """
    s = str(val).strip()
    if s in NON_FINISHES or s in BAD_VALUES:
        return 999, False
    try:
        r = int(float(s))
        return r, True
    except (ValueError, TypeError):
        return 999, False

cleaned = race_results["Rank"].apply(clean_rank)
race_results["Rank"]        = cleaned.apply(lambda x: x[0])
race_results["Did_Finish"]  = cleaned.apply(lambda x: x[1])
race_results["Top10"]       = ((race_results["Rank"] <= 10) & race_results["Did_Finish"]).astype(int)
race_results["Top3"]        = ((race_results["Rank"] <= 3)  & race_results["Did_Finish"]).astype(int)

print("Rank distribution (cleaned):")
print(race_results["Rank"].describe())
print(f"DNF/DNS/etc rows: {(~race_results['Did_Finish']).sum():,}")
print(f"Top 10 finishes:  {race_results['Top10'].sum():,}")
print(f"Top 3 finishes:   {race_results['Top3'].sum():,}")

Rank distribution (cleaned):
count    140573.000000
mean        123.417598
std         208.596843
min           1.000000
25%          38.000000
50%          78.000000
75%         120.000000
max         999.000000
Name: Rank, dtype: float64
DNF/DNS/etc rows: 7,216
Top 10 finishes:  9,621
Top 3 finishes:   2,887


In [11]:
before_rr = len(race_results)
before_cd = len(course_data)

merged_df = pd.merge(
    race_results, course_data,
    on="Race Name", how="inner",
    suffixes=("_results", "_course")
)

lost = before_rr - len(merged_df)
print(f"race_results before merge: {before_rr:,}")
print(f"Rows in merged result:     {len(merged_df):,}")
print(f"Lost in inner join:        {lost:,} ({lost/before_rr*100:.1f}%)")

unmatched = race_results[~race_results["Race Name"].isin(set(merged_df["Race Name"]))]
print(f"Unmatched race names (sample):")
print(unmatched["Race Name"].value_counts().head(15).to_string())

race_results before merge: 140,573
Rows in merged result:     140,573
Lost in inner join:        0 (0.0%)
Unmatched race names (sample):
Series([], )


In [13]:
print("=== FINAL DATASET ===")
print(f"Shape: {merged_df.shape}")
print(f"Years: {sorted(merged_df['Year_results'].unique())}")
print(f"Unique race names: {merged_df['Race Name'].nunique()}")
print(f"Top races by row count:")
print(merged_df["Race Name"].value_counts().head(15).to_string())
print(f"Column null rates:")
for col in merged_df.columns:
    pct = merged_df[col].isna().mean() * 100
    print(f"  {col:<45} {pct:.1f}%")

=== FINAL DATASET ===
Shape: (140573, 44)
Years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Unique race names: 896
Top races by row count:
Race Name
2017 La Fleche Wallonne                    200
2017 Liege - Bastogne - Liege              200
2017 Volta Ciclista a Catalunya Stage 1    200
2017 Il Lombardia                          199
2017 Paris - Roubaix                       199
2017 Ronde Van Vlaanderen                  199
2017 Tour de France Stage 1                198
2017 Vuelta a Espana Stage 1               198
2017 Vuelta a Espana Stage 2               198
2017 Bretagne Classic - Ouest France       197
2017 Giro d'Italia Stage 1                 197
2017 Tour de France Stage 2                196
2017 Volta Ciclista a Catalunya Stage 2    196
2017 Volta Ciclista a Catalunya Stage 3    196
2017 Giro d'Italia Stage 2                 195
Column null rates:
  Race Name                                     0.0%
  Da

In [14]:
merged_df.to_csv(PROCESSED_DATA / "merged_uci_races.csv",    index=False)
race_results.to_csv(PROCESSED_DATA / "race_results_clean.csv", index=False)
course_data.to_csv(PROCESSED_DATA  / "course_data_clean.csv",  index=False)

print("Saved:")
print(f"  merged_uci_races.csv     {len(merged_df):,} rows")
print(f"  race_results_clean.csv   {len(race_results):,} rows")
print(f"  course_data_clean.csv    {len(course_data):,} rows")

Saved:
  merged_uci_races.csv     140,573 rows
  race_results_clean.csv   140,573 rows
  course_data_clean.csv    963 rows
